# PCC Research Question Nanopublication Creator (Multi-PCC Version)

Creates **multiple** PCC nanopublications from a single JSON configuration file.

**Template:** [Defining a PCC-based research question](https://w3id.org/np/RAmR-xqMgOq3oTJmOVDQFL2p5usID6zqRapizHy0UJb04)

This template uses the PCC framework (Population, Concept, Context) for scoping reviews, data papers, and qualitative research.

---

## Instructions

1. **Create a JSON file** with your PCC details (see template at bottom)
2. **Set the path** to your JSON file in Section 1
3. **Run All Cells** → Get multiple `.trig` files (one per PCC question)

---
# 📁 SECTION 1: INPUT FILE (EDIT THIS)
---

In [55]:
# Path to your PCC JSON file (with multiple nanopublications)
CONFIG_FILE = "../biodiversity/crete/crete_declaration_pcc_all.json"
CONFIG_FILE = "../biodiversity/dimuri2022/dimuri2022_pcc.json"
CONFIG_FILE = "../biodiversity/dimuri2023/dimuri2023_pcc.json"
CONFIG_FILE = "../biodiversity/gep/lifewatch_gep_pcc.json"
CONFIG_FILE = "../biodiversity/heraklion/heraklion_pcc.json"

# Output directory for .trig files
OUTPUT_DIR = "./biodiversity/"

---
# ⚙️ SECTION 2: SETUP
---

In [56]:
# Install dependencies (uncomment if needed)
# !pip install nanopub rdflib

In [57]:
import json
import re
import os
from rdflib import Graph, Dataset, Namespace, Literal, URIRef
from rdflib.namespace import RDF, RDFS, XSD, FOAF
from datetime import datetime, timezone
from pathlib import Path

# Namespaces (matching Nanodash)
NP = Namespace("http://www.nanopub.org/nschema#")
DCT = Namespace("http://purl.org/dc/terms/")
NT = Namespace("https://w3id.org/np/o/ntemplate/")
NPX = Namespace("http://purl.org/nanopub/x/")
PROV = Namespace("http://www.w3.org/ns/prov#")
ORCID = Namespace("https://orcid.org/")

# Science Live ontology for PCC
SLV = Namespace("https://w3id.org/sciencelive/o/terms/")

# PCC template URI
PCC_TEMPLATE = URIRef("https://w3id.org/np/RAmR-xqMgOq3oTJmOVDQFL2p5usID6zqRapizHy0UJb04")

# Template references
PROV_TEMPLATE = URIRef("https://w3id.org/np/RA7lSq6MuK_TIC6JMSHvLtee3lpLoZDOqLJCLXevnrPoU")
PUBINFO_TEMPLATE_1 = URIRef("https://w3id.org/np/RA0J4vUn_dekg-U1kK3AOEt02p9mT2WO03uGxLDec1jLw")
PUBINFO_TEMPLATE_2 = URIRef("https://w3id.org/np/RAukAcWHRDlkqxk7H2XNSegc1WnHI569INvNr-xdptDGI")

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✓ Setup complete")

✓ Setup complete


---
# 📖 SECTION 3: LOAD & VALIDATE
---

In [58]:
# Load PCC from JSON
print(f"Loading: {CONFIG_FILE}")

with open(CONFIG_FILE, 'r', encoding='utf-8') as f:
    config = json.load(f)

# Extract metadata
AUTHOR_ORCID = config['metadata']['creator_orcid']
AUTHOR_NAME = config['metadata']['creator_name']
FILENAME_PREFIX = config.get('output', {}).get('filename_prefix', 'pcc')

# Get source paper info if available
SOURCE_PAPER = config['metadata'].get('source_paper', {})

# Get all PCC questions
pcc_list = config['nanopublications']

print(f"✓ Loaded {len(pcc_list)} PCC questions")
print(f"  Author: {AUTHOR_NAME} ({AUTHOR_ORCID})")
if SOURCE_PAPER:
    print(f"  Source: {SOURCE_PAPER.get('title', 'N/A')}")
print()
for i, pcc in enumerate(pcc_list, 1):
    print(f"  {i}. {pcc['label'][:60]}..." if len(pcc['label']) > 60 else f"  {i}. {pcc['label']}")

Loading: ../biodiversity/heraklion/heraklion_pcc.json
✓ Loaded 1 PCC questions
  Author: Anne Fouilloux (0000-0002-1784-2920)
  Source: Macrobenthic communities of the continental shelf of Heraklion Bay (Crete, Greece): bathymetric distribution and temporal trends

  1. Bathymetric and temporal patterns of macrobenthic communitie...


In [59]:
# Validate all PCC entries
print("Validating...")

all_errors = []

# Check metadata
if not AUTHOR_ORCID or AUTHOR_ORCID == "0000-0000-0000-0000":
    all_errors.append("metadata.creator_orcid must be set to your real ORCID")
if not AUTHOR_NAME or AUTHOR_NAME == "Your Name":
    all_errors.append("metadata.creator_name must be set to your real name")

# Check each PCC
for i, pcc in enumerate(pcc_list):
    prefix = f"nanopublications[{i}]"
    if not pcc.get('id'):
        all_errors.append(f"{prefix}.id is required")
    if not pcc.get('label') or len(pcc.get('label', '')) < 10:
        all_errors.append(f"{prefix}.label must be at least 10 characters")
    if not pcc.get('population', {}).get('description'):
        all_errors.append(f"{prefix}.population.description is required")
    if not pcc.get('concept', {}).get('description'):
        all_errors.append(f"{prefix}.concept.description is required")
    if not pcc.get('context', {}).get('description'):
        all_errors.append(f"{prefix}.context.description is required")
    if not pcc.get('description'):
        all_errors.append(f"{prefix}.description (research question) is required")

if all_errors:
    print("❌ Validation errors:")
    for e in all_errors:
        print(f"   - {e}")
    raise ValueError("Please fix the errors in your JSON file")
else:
    print("✓ All fields valid")

Validating...
✓ All fields valid


---
# 🔨 SECTION 4: BUILD NANOPUBLICATIONS
---

In [60]:
def create_pcc_nanopub(pcc_data, author_orcid, author_name):
    """
    Create a single PCC nanopublication from PCC data.
    
    Returns: (Dataset, output_filename)
    """
    # Extract PCC fields
    PCC_ID = pcc_data['id']
    TITLE = pcc_data['label']
    RESEARCH_QUESTION = pcc_data['description']
    POPULATION = pcc_data['population']['description']
    CONCEPT = pcc_data['concept']['description']
    CONTEXT = pcc_data['context']['description']
    
    # Create dataset with named graphs
    TEMP_NP = Namespace("http://purl.org/nanopub/temp/np/")
    
    this_np = URIRef("http://purl.org/nanopub/temp/np/")
    head_graph = URIRef("http://purl.org/nanopub/temp/np/Head")
    assertion_graph = URIRef("http://purl.org/nanopub/temp/np/assertion")
    provenance_graph = URIRef("http://purl.org/nanopub/temp/np/provenance")
    pubinfo_graph = URIRef("http://purl.org/nanopub/temp/np/pubinfo")
    
    ds = Dataset()
    
    # Bind prefixes
    ds.bind("this", "http://purl.org/nanopub/temp/np/")
    ds.bind("sub", TEMP_NP)
    ds.bind("np", NP)
    ds.bind("dct", DCT)
    ds.bind("nt", NT)
    ds.bind("npx", NPX)
    ds.bind("xsd", XSD)
    ds.bind("rdfs", RDFS)
    ds.bind("orcid", ORCID)
    ds.bind("prov", PROV)
    ds.bind("foaf", FOAF)
    ds.bind("slv", SLV)
    
    # HEAD
    head = ds.graph(head_graph)
    head.add((this_np, RDF.type, NP.Nanopublication))
    head.add((this_np, NP.hasAssertion, assertion_graph))
    head.add((this_np, NP.hasProvenance, provenance_graph))
    head.add((this_np, NP.hasPublicationInfo, pubinfo_graph))
    
    # ASSERTION
    assertion = ds.graph(assertion_graph)
    
    # Create local resource URIs
    pcc_uri = TEMP_NP[PCC_ID]
    population_uri = TEMP_NP["population"]
    concept_uri = TEMP_NP["concept"]
    context_uri = TEMP_NP["context"]
    
    # Main PCC resource
    assertion.add((pcc_uri, RDF.type, SLV.PccReviewQuestion))
    assertion.add((pcc_uri, RDFS.label, Literal(TITLE)))
    assertion.add((pcc_uri, DCT.description, Literal(RESEARCH_QUESTION)))
    
    # Population
    assertion.add((pcc_uri, SLV.hasPccPopulation, population_uri))
    assertion.add((population_uri, DCT.description, Literal(POPULATION)))
    
    # Concept
    assertion.add((pcc_uri, SLV.hasPccConcept, concept_uri))
    assertion.add((concept_uri, DCT.description, Literal(CONCEPT)))
    
    # Context
    assertion.add((pcc_uri, SLV.hasPccContext, context_uri))
    assertion.add((context_uri, DCT.description, Literal(CONTEXT)))
    
    # PROVENANCE
    provenance = ds.graph(provenance_graph)
    author_uri = ORCID[author_orcid]
    provenance.add((assertion_graph, PROV.wasAttributedTo, author_uri))
    
    # PUBINFO
    pubinfo = ds.graph(pubinfo_graph)
    now = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")
    
    pubinfo.add((author_uri, FOAF.name, Literal(author_name)))
    pubinfo.add((this_np, DCT.created, Literal(now, datatype=XSD.dateTime)))
    pubinfo.add((this_np, DCT.creator, author_uri))
    pubinfo.add((this_np, DCT.license, URIRef("https://creativecommons.org/licenses/by/4.0/")))
    pubinfo.add((this_np, NPX.wasCreatedAt, URIRef("https://nanodash.knowledgepixels.com/")))
    pubinfo.add((this_np, NPX.introduces, pcc_uri))
    
    # Template references
    pubinfo.add((this_np, NT.wasCreatedFromTemplate, PCC_TEMPLATE))
    pubinfo.add((this_np, NT.wasCreatedFromProvenanceTemplate, PROV_TEMPLATE))
    pubinfo.add((this_np, NT.wasCreatedFromPubinfoTemplate, PUBINFO_TEMPLATE_1))
    pubinfo.add((this_np, NT.wasCreatedFromPubinfoTemplate, PUBINFO_TEMPLATE_2))
    
    return ds, PCC_ID

print("✓ Function defined")

✓ Function defined


In [61]:
# Build all nanopublications
print(f"Building {len(pcc_list)} nanopublications...")
print("=" * 70)

created_files = []

for i, pcc in enumerate(pcc_list, 1):
    # Create nanopub
    ds, pcc_id = create_pcc_nanopub(pcc, AUTHOR_ORCID, AUTHOR_NAME)
    
    # Serialize to TriG
    trig_output = ds.serialize(format='trig')
    
    # Save to file
    output_filename = f"{FILENAME_PREFIX}_{pcc_id}.trig"
    output_path = Path(OUTPUT_DIR) / output_filename
    
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(trig_output)
    
    created_files.append(output_path)
    print(f"  {i}. ✓ {output_filename}")
    print(f"       {pcc['label'][:55]}..." if len(pcc['label']) > 55 else f"       {pcc['label']}")

print("=" * 70)
print(f"✓ Created {len(created_files)} nanopublication files in {OUTPUT_DIR}")

Building 1 nanopublications...
  1. ✓ heraklion_pcc_pcc_heraklion_macrobenthos.trig
       Bathymetric and temporal patterns of macrobenthic commu...
✓ Created 1 nanopublication files in ./biodiversity/


---
# 💾 SECTION 5: SUMMARY
---

In [62]:
# Summary
print("=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"Input:     {CONFIG_FILE}")
print(f"Output:    {OUTPUT_DIR}")
print(f"Author:    {AUTHOR_NAME} (orcid:{AUTHOR_ORCID})")
print(f"Template:  {PCC_TEMPLATE}")
print()
print(f"Created {len(created_files)} PCC nanopublications:")
print()
for i, (pcc, path) in enumerate(zip(pcc_list, created_files), 1):
    print(f"  {i}. {path.name}")
    print(f"     P: {pcc['population']['description'][:50]}...")
    print(f"     C: {pcc['concept']['description'][:50]}...")
    print(f"     C: {pcc['context']['description'][:50]}...")
    print()

print("Next steps:")
print(f"  Sign all:    for f in {OUTPUT_DIR}*.trig; do nanopub sign \"$f\"; done")
print(f"  Publish all: for f in {OUTPUT_DIR}*.signed.trig; do nanopub publish \"$f\"; done")

SUMMARY
Input:     ../biodiversity/heraklion/heraklion_pcc.json
Output:    ./biodiversity/
Author:    Anne Fouilloux (orcid:0000-0002-1784-2920)
Template:  https://w3id.org/np/RAmR-xqMgOq3oTJmOVDQFL2p5usID6zqRapizHy0UJb04

Created 1 PCC nanopublications:

  1. heraklion_pcc_pcc_heraklion_macrobenthos.trig
     P: Macrobenthic faunal communities including Polychae...
     C: Taxon composition, abundance, bathymetric distribu...
     C: Continental shelf of Heraklion Bay (Crete, Greece)...

Next steps:
  Sign all:    for f in ./biodiversity/*.trig; do nanopub sign "$f"; done
  Publish all: for f in ./biodiversity/*.signed.trig; do nanopub publish "$f"; done


---
# 🚀 SECTION 6: SIGN & PUBLISH (OPTIONAL)
---

In [63]:
PUBLISH = True
USE_TEST_SERVER = False
PROFILE_PATH = "/Users/annef/Documents/ScienceLive/ai-profile/profile.yml" 

In [64]:
if PUBLISH:
    from nanopub import Nanopub, NanopubConf, load_profile
    
    if PROFILE_PATH:
        profile = load_profile(PROFILE_PATH)
    else:
        profile = load_profile()
    print(f"Loaded profile: {profile.name}")
    print("=" * 70)
    
    conf = NanopubConf(profile=profile, use_test_server=USE_TEST_SERVER)
    published_uris = []
    
    for i, trig_path in enumerate(created_files, 1):
        print(f"\n[{i}/{len(created_files)}] Processing {trig_path.name}...")
        
        np_obj = Nanopub(rdf=trig_path, conf=conf)
        
        np_obj.sign()
        print(f"  ✓ Signed")
        
        signed_path = trig_path.with_suffix('.signed.trig')
        np_obj.store(signed_path)
        print(f"  ✓ Saved: {signed_path.name}")
        
        np_obj.publish()
        print(f"  ✓ Published: {np_obj.source_uri}")
        published_uris.append(np_obj.source_uri)
    
    print("\n" + "=" * 70)
    print(f"✓ Published {len(published_uris)} nanopublications:")
    for uri in published_uris:
        print(f"  {uri}")
else:
    print("Publishing disabled. Set PUBLISH = True to enable.")

Loaded profile: claude-ai-agent

[1/1] Processing heraklion_pcc_pcc_heraklion_macrobenthos.trig...
  ✓ Signed
  ✓ Saved: heraklion_pcc_pcc_heraklion_macrobenthos.signed.trig
  ✓ Published: https://w3id.org/np/RAjWlL-ibJbhx2aEaHtfYe-ZI_sI10yfptgCQTJpc5M0c

✓ Published 1 nanopublications:
  https://w3id.org/np/RAjWlL-ibJbhx2aEaHtfYe-ZI_sI10yfptgCQTJpc5M0c


---
# 📋 JSON TEMPLATE (Multi-PCC)

Create a JSON file with this structure:

```json
{
  "metadata": {
    "creator_orcid": "0000-0000-0000-0000",
    "creator_name": "Your Name",
    "source_paper": {
      "title": "Paper Title",
      "doi": "https://doi.org/10.xxxx/xxxxx"
    }
  },
  "nanopublications": [
    {
      "id": "pcc_question_1",
      "label": "Short title for question 1",
      "description": "Full research question 1?",
      "population": {
        "description": "Who or what is being studied"
      },
      "concept": {
        "description": "What concept/phenomenon is explored"
      },
      "context": {
        "description": "Setting, time period, geography"
      }
    },
    {
      "id": "pcc_question_2",
      "label": "Short title for question 2",
      "description": "Full research question 2?",
      "population": {
        "description": "..."
      },
      "concept": {
        "description": "..."
      },
      "context": {
        "description": "..."
      }
    }
  ],
  "output": {
    "filename_prefix": "my_project_pcc"
  }
}
```

---

## PCC vs PICO

| Component | PICO | PCC |
|-----------|------|-----|
| P | Population | Population |
| I | Intervention | - |
| C | Comparison | Concept |
| O | Outcome | Context |

Use **PCC** for:
- Scoping reviews
- Data papers (describing/documenting data)
- Qualitative research
- Descriptive studies
- Mapping/charting exercises

Use **PICO** for:
- Clinical trials
- Intervention studies
- Effectiveness research

---